In [ ]:
import numpy
import torchvision
import torchvision.transforms.v2

DATASET = "E:/DiplomaV2/full_run_2/gtsrb/results"

transform = torchvision.transforms.v2.Resize((32, 32), antialias=True)
train = torchvision.datasets.GTSRB('gtsrb', split = 'train', transform = transform, download = True)
test = torchvision.datasets.GTSRB('gtsrb', split = 'test', transform = transform, download = True)

train_images = numpy.array([numpy.array(item[0]) / 255 for item in train])
train_labels = numpy.array([item[1] for item in train])

test_images = numpy.array([numpy.array(item[0]) / 255 for item in test])
test_labels = numpy.array([item[1] for item in test])

In [2]:
import cvtda.topology

fe = cvtda.topology.FeatureExtractor(n_jobs = 1)
train_features = fe.fit_transform(train_images, dump_name = f'{DATASET}/train')
test_features = fe.transform(test_images, dump_name = f'{DATASET}/test')

RGB images received. Transforming to grayscale.


rgb2gray: 100%|██████████| 26640/26640 [00:00<00:00, 40009.10it/s]


GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/gray, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/gray/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting complete
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/gray/features.npy
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/red, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/red/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting complete
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/red/features.npy
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/green, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/greyscale/green/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting comp

rgb2gray: 100%|██████████| 26640/26640 [00:00<00:00, 44030.21it/s]


GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/gray, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/gray/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting complete
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/gray/features.npy
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/red, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/red/diagrams.npy
Applying Scaler to persistence diagrams.
DiagramVectorizer: fitting complete
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/red/features.npy
GreyscaleExtractor: processing E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/green, do_fit = True
Got the result from E:/DiplomaV2/full_run_2/gtsrb/results/train/inverted_greyscale/green/diagrams.npy


 49%|████▊     | 35/72 [45:26<48:02, 77.90s/it]   


KeyboardInterrupt: 

In [ ]:
import cvtda.utils
import sklearn.tree
import cvtda.topology
import sklearn.metrics
import cvtda.classification

def explain_decision_tree(fe: cvtda.topology.FeatureExtractor, dt: sklearn.tree.DecisionTreeClassifier, obj: int):
    target, prediction = test_labels[obj], dt.predict([test_features[obj]])[0]
    match = "✓" if target == prediction else "✗"

    path = dt.decision_path([test_features[obj]]).toarray()[0]
    features_idxs = dt.tree_.feature[numpy.where(path != 0)[0][:-1]]
    feature_names = list(numpy.array(fe.feature_names())[features_idxs])
    return cvtda.utils.FeatureExplanation.display_many(
        [fe.explain(f, test_images[obj]) for f in feature_names],
        title=f"Object {obj}  ·  target: {target}   prediction: {prediction} {match}",
    )

dt = sklearn.tree.DecisionTreeClassifier(max_depth = 8, max_features = 1/3, random_state = 42)
dt.fit(train_features, train_labels)
cvtda.classification.estimate_quality(dt.predict_proba(test_features), test_labels)

In [ ]:
import os
import tqdm
import joblib
import matplotlib.pyplot as plt

def process_item(i):
    plt.rcParams.update({ 'font.size': 13 })
    fig = explain_decision_tree(fe, dt, i)
    y = test_labels[i]
    os.makedirs(f"cvtda_results/gtsrb/{y}", exist_ok=True)
    fig.savefig(f"cvtda_results/gtsrb/{y}/{i}.png")
    fig.savefig(f"cvtda_results/gtsrb/{y}/{i}.svg")
    plt.close(fig)

joblib.Parallel(n_jobs=-1)(joblib.delayed(process_item)(i) for i in tqdm.trange(len(test)))
None